# Inform 27 — Importação simples para SQLite

Fluxo:

1. Encontrar a pasta dos ficheiros.
2. Ler os `.xls`.
3. Validar `FENTREGA` e obter o ano.
4. Criar a tabela desse ano.
5. Inserir por `CODEDT`, sem duplicados.
6. Mostrar um resumo final.

A lógica de caminhos não depende do nome do utilizador.


In [1]:
import os
import platform
import sqlite3
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime
import re

COLUNAS = [
    "PROPIETARIO", "TRAYECTO", "TRANSPORTISTA", "TRACTORA", "REMOLQUE",
    "INGRESODT", "COSTEDT", "RENTADT", "PALETSDT", "PESO_BRUTO",
    "CODEUT", "ESTADO_UT", "RANGO_UT", "FCARGA", "ACTIVIDAD",
    "CODEDT", "ESTADO_DT", "REFERENCIA", "CODACT", "LOCORIGEN",
    "PROV_ORIGEN", "PAISORIGEN", "CPOSTAL", "LOCDESTINO", "PROV_DESTINO",
    "PAISDESTINO", "CPOSTAD", "KM", "FENTREGA", "ORIGEN",
    "ENTREGAR", "PROV_ENTREGAR", "PAISENTREGAR", "DESTINO", "PALETS",
    "PREFAC", "RUTA", "COBROREAL", "GESTION", "DEPART",
    "USCODE", "USUARIO", "TIPOCLIENTE", "TIPOFLUJO", "WMSCODRGT",
    "LOCCAR", "LUGARCARGA", "LOCDES", "LUGARDESCARGA", "TEMP_MERC_PED",
    "TIPOPALETA", "CAMION_TIPO", "CAMION_CAPACIDAD", "TIPO_COMBUSTIBLE",
    "KMREALES", "ALBARAN"
]

COLUNA_DATA = "FENTREGA"
COLUNA_CHAVE = "CODEUT"
COLUNA_ORIGEM = "ficheiro_origem"

NS = {"ss": "urn:schemas-microsoft-com:office:spreadsheet"}
SS_INDEX = "{urn:schemas-microsoft-com:office:spreadsheet}Index"

In [2]:
if platform.system() == 'Windows':
    DB_PATH = r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db"
    PASTA_FICHEIROS = Path(r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27")
elif platform.system() == 'Darwin':
    DB_PATH = "/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/inform_27.db"
    PASTA_FICHEIROS = Path("/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/2026_dados")
else:
    DB_PATH = "inform_27.db"
    PASTA_FICHEIROS = Path("inform_27")

# Criar pasta se não existir
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

# Criar BD
con = sqlite3.connect(DB_PATH)
con.close()
print(f"✓ BD: {DB_PATH}")

✓ BD: /Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/inform_27.db


In [3]:
# ==========================
# 2. Leitura e validação
# ==========================
def ler_xls(caminho):
    tree = ET.parse(caminho)
    tabela = tree.getroot().find(".//ss:Table", NS)

    if tabela is None:
        raise ValueError("Tabela XML não encontrada.")

    def ler_linha(row):
        valores = []
        proximo = 1

        for cell in row.findall("ss:Cell", NS):
            indice = cell.get(SS_INDEX)
            indice = int(indice) if indice else proximo

            while len(valores) < indice - 1:
                valores.append(None)

            data = cell.find("ss:Data", NS)
            valores.append(data.text if data is not None else None)
            proximo = indice + 1

        return valores

    rows = tabela.findall("ss:Row", NS)

    if not rows:
        return [], []

    cabecalho = ler_linha(rows[0])
    cabecalho = [
        str(valor).strip() if valor else f"COLUNA_{i + 1}"
        for i, valor in enumerate(cabecalho)
    ]

    linhas = []
    for row in rows[1:]:
        valores = ler_linha(row)

        if len(valores) < len(cabecalho):
            valores += [None] * (len(cabecalho) - len(valores))

        linhas.append(valores[:len(cabecalho)])

    return cabecalho, linhas


def obter_ano(valor):
    if valor is None:
        return None

    texto = str(valor).strip()

    formatos = [
        "%Y%m%d",
        "%Y-%m-%d",
        "%Y/%m/%d",
    ]

    for formato in formatos:
        try:
            return datetime.strptime(texto[:10], formato).year
        except ValueError:
            pass

    match = re.match(r"^(\d{4})\d{4}", texto)
    if match and match.group(1) != "0000":
        return int(match.group(1))

    return None


In [5]:
# ==========================
# 3. Base de dados
# ==========================
def tabela_ano(ano):
    return f"inform_27_{ano}"

def criar_tabela(con, ano):
    tabela = tabela_ano(ano)

    colunas_sql = ", ".join(
        f'"{coluna}" TEXT' for coluna in COLUNAS
    )

    con.execute(f"""
        CREATE TABLE IF NOT EXISTS "{tabela}" (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            {colunas_sql},
            "{COLUNA_ORIGEM}" TEXT
        )
    """)

    con.execute(
    f'CREATE UNIQUE INDEX IF NOT EXISTS "idx_{tabela}_codedt_fentrega_codeut" '
    f'ON "{tabela}" ("CODEDT", "FENTREGA", "CODEUT")'
    )

    return tabela

def inserir_linhas(con, ano, linhas):
    if not linhas:
        return 0, 0

    tabela = criar_tabela(con, ano)

    colunas_sql = ", ".join(f'"{coluna}"' for coluna in COLUNAS)
    placeholders = ", ".join("?" for _ in range(len(COLUNAS) + 1))

    sql = (
        f'INSERT OR IGNORE INTO "{tabela}" '
        f'({colunas_sql}, "{COLUNA_ORIGEM}") '
        f"VALUES ({placeholders})"
    )

    antes = con.total_changes
    con.executemany(sql, linhas)
    inseridos = con.total_changes - antes

    return inseridos, len(linhas) - inseridos


In [6]:
# ==========================
# 4. Importação
# ==========================
ficheiros = sorted(PASTA_FICHEIROS.rglob("*.xls"))

if not ficheiros:
    raise FileNotFoundError(
        f"Nenhum ficheiro .xls encontrado em: {PASTA_FICHEIROS}"
    )

con = sqlite3.connect(DB_PATH)

totais = {
    "ficheiros": 0,
    "linhas": 0,
    "inseridos": {},
    "duplicados": 0,
    "sem_data": 0,
    "erros": 0,
}

for numero, caminho in enumerate(ficheiros, 1):
    try:
        cabecalho, linhas = ler_xls(caminho)

        if COLUNA_DATA not in cabecalho:
            print(f"[{numero}/{len(ficheiros)}] {caminho.name} — sem {COLUNA_DATA}")
            totais["erros"] += 1
            continue

        mapa_indices = {
            coluna: i for i, coluna in enumerate(cabecalho)
        }

        linhas_por_ano = {}
        sem_data = 0

        for valores in linhas:
            mapa = {
                coluna: valores[i] if i < len(valores) else None
                for coluna, i in mapa_indices.items()
            }

            ano = obter_ano(mapa.get(COLUNA_DATA))

            if ano is None:
                sem_data += 1
                continue

            linha = [mapa.get(coluna) for coluna in COLUNAS]
            linha.append(caminho.name)

            linhas_por_ano.setdefault(ano, []).append(linha)

        novos_ficheiro = 0
        duplicados_ficheiro = 0

        for ano, batch in sorted(linhas_por_ano.items()):
            inseridos, duplicados = inserir_linhas(con, ano, batch)

            totais["inseridos"][ano] = (
                totais["inseridos"].get(ano, 0) + inseridos
            )

            novos_ficheiro += inseridos
            duplicados_ficheiro += duplicados

        totais["ficheiros"] += 1
        totais["linhas"] += len(linhas)
        totais["duplicados"] += duplicados_ficheiro
        totais["sem_data"] += sem_data

        con.commit()

        print(
            f"[{numero}/{len(ficheiros)}] {caminho.name} | "
            f"+{novos_ficheiro:,} novos | "
            f"{duplicados_ficheiro:,} duplicados | "
            f"{sem_data:,} sem data"
        )

    except Exception as erro:
        con.rollback()
        totais["erros"] += 1
        print(f"[{numero}/{len(ficheiros)}] {caminho.name} — ERRO: {erro}")

con.close()

print("\n--- RESUMO ---")
print(f"Ficheiros processados: {totais['ficheiros']}")
print(f"Linhas lidas: {totais['linhas']:,}")
print(f"Linhas duplicadas: {totais['duplicados']:,}")
print(f"Linhas sem FENTREGA válida: {totais['sem_data']:,}")
print(f"Ficheiros com erro: {totais['erros']}")

for ano, quantidade in sorted(totais["inseridos"].items()):
    print(f"Linhas novas em {ano}: {quantidade:,}")


[1/8] SAL_DAT027 (1).xls | +0 novos | 65,534 duplicados | 0 sem data
[2/8] SAL_DAT027 (2).xls | +0 novos | 65,534 duplicados | 0 sem data
[3/8] SAL_DAT027 (3).xls | +0 novos | 65,526 duplicados | 8 sem data
[4/8] SAL_DAT027 (4).xls | +0 novos | 65,534 duplicados | 0 sem data
[5/8] SAL_DAT027 (5).xls | +0 novos | 65,529 duplicados | 5 sem data
[6/8] SAL_DAT027 (6).xls | +0 novos | 65,528 duplicados | 6 sem data
[7/8] SAL_DAT027 (7).xls | +0 novos | 59,349 duplicados | 21 sem data
[8/8] SAL_DAT027.xls | +0 novos | 65,530 duplicados | 4 sem data

--- RESUMO ---
Ficheiros processados: 8
Linhas lidas: 518,108
Linhas duplicadas: 518,064
Linhas sem FENTREGA válida: 44
Ficheiros com erro: 0
Linhas novas em 2025: 0
Linhas novas em 2026: 0


In [6]:
# ==========================
# 5. Verificação da BD
# ==========================
con = sqlite3.connect(DB_PATH)

tabelas = [
    row[0]
    for row in con.execute(
        "SELECT name FROM sqlite_master "
        "WHERE type='table' AND name LIKE 'inform_27_%' "
        "ORDER BY name"
    )
]

print("Tabelas de dados:")
print("\n".join(f"  - {tabela}" for tabela in tabelas))
print()

total = 0

for tabela in tabelas:
    total_ano = con.execute(
        f'SELECT COUNT(*) FROM "{tabela}"'
    ).fetchone()[0]

    total += total_ano

    print(f"{tabela}: {total_ano:,} registos")

print(f"\nTOTAL: {total:,}")

con.close()


Tabelas de dados:
  - inform_27_2025
  - inform_27_2026

inform_27_2025: 15,418 registos
inform_27_2026: 502,410 registos

TOTAL: 517,828


# Validações


def validar_chave_unica(ano):
    """Valida se (CODEDT, FENTREGA, CODEUT) é única nos ficheiros e BD."""
    
    ficheiros = sorted(PASTA_FICHEIROS.rglob("*.xls"))
    con = sqlite3.connect(DB_PATH)
    tabela = tabela_ano(ano)
    
    print(f"Validando chave única (CODEDT, FENTREGA, CODEUT) em {ano}\n")
    
    print("="*60)
    print("FICHEIROS")
    print("="*60)
    
    duplicados_ficheiros = 0
    
    for caminho in ficheiros:
        try:
            cabecalho, linhas_ficheiro = ler_xls(caminho)
            mapa_indices = {coluna: i for i, coluna in enumerate(cabecalho)}
            
            assinaturas = []
            
            for valores in linhas_ficheiro:
                data = valores[mapa_indices.get(COLUNA_DATA)]
                if obter_ano(data) != ano:
                    continue
                
                codedt = valores[mapa_indices.get("CODEDT")]
                fentrega = valores[mapa_indices.get("FENTREGA")]
                codeut = valores[mapa_indices.get("CODEUT")]
                
                assinatura = (codedt, fentrega, codeut)
                assinaturas.append(assinatura)
            
            # Detectar duplicados
            assinaturas_unicas = len(set(assinaturas))
            duplicados = len(assinaturas) - assinaturas_unicas
            duplicados_ficheiros += duplicados
            
            if duplicados > 0:
                print(f"  {caminho.name}: ⚠️  {duplicados} duplicados ({len(assinaturas)} totais, {assinaturas_unicas} únicos)")
                
                # Mostrar quais são duplicados
                from collections import Counter
                contagem = Counter(assinaturas)
                duplicados_encontrados = {k: v for k, v in contagem.items() if v > 1}
                for (codedt, fentrega, codeut), count in list(duplicados_encontrados.items())[:3]:
                    print(f"      → CODEDT={codedt}, FENTREGA={fentrega}, CODEUT={codeut}: {count}x")
            else:
                print(f"  {caminho.name}: ✓ OK ({assinaturas_unicas} registos únicos)")
        
        except Exception as erro:
            print(f"  {caminho.name}: ❌ ERRO: {erro}")
    
    print(f"\n{'='*60}")
    print("BASE DE DADOS")
    print("="*60)
    
    # Validar BD
    cursor = con.execute(f"""
        SELECT CODEDT, FENTREGA, CODEUT, COUNT(*) as count
        FROM "{tabela}"
        GROUP BY CODEDT, FENTREGA, CODEUT
        HAVING COUNT(*) > 1
    """)
    
    duplicados_bd = cursor.fetchall()
    
    if duplicados_bd:
        print(f"  ❌ {len(duplicados_bd)} chaves duplicadas na BD:")
        for codedt, fentrega, codeut, count in duplicados_bd[:5]:
            print(f"      → CODEDT={codedt}, FENTREGA={fentrega}, CODEUT={codeut}: {count}x")
    else:
        print(f"  ✓ Nenhuma chave duplicada na BD")
    
    con.close()
    
    print(f"\n{'='*60}")
    print("RESUMO")
    print("="*60)
    print(f"Duplicados nos ficheiros: {duplicados_ficheiros}")
    print(f"Duplicados na BD: {len(duplicados_bd)}")
    
    if duplicados_ficheiros == 0 and len(duplicados_bd) == 0:
        print("\n✓ Chave única validada com sucesso!")
    else:
        print("\n❌ Problemas detectados na integridade da chave única")


# Usar:
validar_chave_unica(2026)

def validar_chave_unica_cruzada(ano):
    """Valida se a chave é única ENTRE todos os ficheiros."""
    
    ficheiros = sorted(PASTA_FICHEIROS.rglob("*.xls"))
    
    print(f"Validando chave única ENTRE ficheiros ({ano})\n")
    
    assinaturas_globais = {}
    duplicados_totais = 0
    
    for caminho in ficheiros:
        try:
            cabecalho, linhas_ficheiro = ler_xls(caminho)
            mapa_indices = {coluna: i for i, coluna in enumerate(cabecalho)}
            
            for valores in linhas_ficheiro:
                data = valores[mapa_indices.get(COLUNA_DATA)]
                if obter_ano(data) != ano:
                    continue
                
                codedt = valores[mapa_indices.get("CODEDT")]
                fentrega = valores[mapa_indices.get("FENTREGA")]
                codeut = valores[mapa_indices.get("CODEUT")]
                
                assinatura = (codedt, fentrega, codeut)
                
                if assinatura in assinaturas_globais:
                    duplicados_totais += 1
                    if duplicados_totais <= 5:
                        ficheiro_anterior = assinaturas_globais[assinatura]
                        print(f"  ❌ Duplicado entre ficheiros:")
                        print(f"      → {ficheiro_anterior} vs {caminho.name}")
                        print(f"      → CODEDT={codedt}, FENTREGA={fentrega}, CODEUT={codeut}")
                else:
                    assinaturas_globais[assinatura] = caminho.name
        
        except Exception as erro:
            print(f"  {caminho.name}: ERRO: {erro}")
    
    print(f"\n{'='*60}")
    if duplicados_totais == 0:
        print(f"✓ Nenhum duplicado entre ficheiros!")
        print(f"  Total de registos únicos: {len(assinaturas_globais):,}")
    else:
        print(f"❌ {duplicados_totais} duplicados encontrados entre ficheiros")
    
    return duplicados_totais


# Usar:
validar_chave_unica_cruzada(2026)

In [10]:
def validar_ficheiros_eficiente(ano):
    """Valida ficheiros vs BD usando assinaturas (CODEDT, FENTREGA, CODEUT)."""
    
    ficheiros = sorted(PASTA_FICHEIROS.rglob("*.xls"))
    con = sqlite3.connect(DB_PATH)
    tabela = tabela_ano(ano)
    
    print(f"Validando {len(ficheiros)} ficheiros contra {tabela}\n")
    
    resumo = {"ok": 0, "faltam": 0, "extras": 0, "erros": 0}
    
    for caminho in ficheiros:
        try:
            # Ler ficheiro
            cabecalho, linhas_ficheiro = ler_xls(caminho)
            mapa_indices = {coluna: i for i, coluna in enumerate(cabecalho)}
            
            # Gerar assinaturas do ficheiro
            assinaturas_ficheiro = set()
            
            for valores in linhas_ficheiro:
                data = valores[mapa_indices.get(COLUNA_DATA)]
                if obter_ano(data) != ano:
                    continue
                
                codedt = valores[mapa_indices.get("CODEDT")]
                fentrega = valores[mapa_indices.get("FENTREGA")]
                codeut = valores[mapa_indices.get("CODEUT")]
                
                assinatura = (codedt, fentrega, codeut)
                assinaturas_ficheiro.add(assinatura)
            
            # Gerar assinaturas da BD (só deste ficheiro)
            cursor = con.execute(
                f'SELECT CODEDT, FENTREGA, CODEUT FROM "{tabela}" WHERE ficheiro_origem = ?',
                (caminho.name,)
            )
            assinaturas_tabela = set(cursor.fetchall())
            
            # Comparar
            faltam = assinaturas_ficheiro - assinaturas_tabela
            extras = assinaturas_tabela - assinaturas_ficheiro
            
            if not faltam and not extras:
                print(f"  {caminho.name}: ✓ OK ({len(assinaturas_ficheiro)} registos)")
                resumo["ok"] += 1
            else:
                print(f"  {caminho.name}: ❌")
                if faltam:
                    print(f"    → {len(faltam)} registos faltam na BD")
                    resumo["faltam"] += len(faltam)
                if extras:
                    print(f"    → {len(extras)} registos extras na BD")
                    resumo["extras"] += len(extras)
        
        except Exception as erro:
            print(f"  {caminho.name}: ❌ ERRO: {erro}")
            resumo["erros"] += 1
    
    con.close()
    
    print(f"\n{'='*50}")
    print(f"Ficheiros OK: {resumo['ok']}")
    print(f"Registos faltam: {resumo['faltam']}")
    print(f"Registos extras: {resumo['extras']}")
    print(f"Erros: {resumo['erros']}")


# Usar:
validar_ficheiros_eficiente(2026)

Validando 8 ficheiros contra inform_27_2026

  SAL_DAT027 (1).xls: ✓ OK (65534 registos)
  SAL_DAT027 (2).xls: ✓ OK (65534 registos)
  SAL_DAT027 (3).xls: ✓ OK (65526 registos)
  SAL_DAT027 (4).xls: ✓ OK (65534 registos)
  SAL_DAT027 (5).xls: ❌
    → 117 registos faltam na BD
  SAL_DAT027 (6).xls: ❌
    → 119 registos faltam na BD
  SAL_DAT027 (7).xls: ✓ OK (59349 registos)
  SAL_DAT027.xls: ✓ OK (50112 registos)

Ficheiros OK: 6
Registos faltam: 236
Registos extras: 0
Erros: 0


def validar_homogeneidade_colunas(ano):
    """Verifica se cada coluna tem dados homogéneos."""
    
    con = sqlite3.connect(DB_PATH)
    tabela = tabela_ano(ano)
    
    cursor = con.execute(f'SELECT * FROM "{tabela}"')
    dados = cursor.fetchall()
    con.close()
    
    if not dados:
        print("Tabela vazia!")
        return
    
    print(f"Validando homogeneidade em {len(dados)} registos...\n")
    
    for i, coluna in enumerate(COLUNAS):
        valores = [linha[i+1] for linha in dados]  # [i+1] ignora ID
        valores = [v for v in valores if v is not None and str(v).strip() != ""]
        
        if not valores:
            print(f"  {coluna}: ⚠️  Vazio ou NULL")
            continue
        
        # Detectar tipo
        tipos = {"int": 0, "float": 0, "data": 0, "texto": 0}
        
        for valor in valores:
            texto = str(valor).strip()
            
            try:
                int(texto)
                tipos["int"] += 1
            except ValueError:
                try:
                    float(texto)
                    tipos["float"] += 1
                except ValueError:
                    if re.match(r"^\d{4}[-/]\d{2}[-/]\d{2}$|^\d{8}$", texto):
                        tipos["data"] += 1
                    else:
                        tipos["texto"] += 1
        
        # Determinar tipo dominante
        tipo_dominante = max(tipos, key=tipos.get)
        percentagem = (tipos[tipo_dominante] / len(valores)) * 100
        
        if percentagem < 90:
            print(f"  {coluna}: ⚠️  {percentagem:.0f}% {tipo_dominante} (misto!)")
            print(f"    → Int: {tipos['int']}, Float: {tipos['float']}, Data: {tipos['data']}, Texto: {tipos['texto']}")
        else:
            print(f"  {coluna}: ✓ {percentagem:.0f}% {tipo_dominante}")


# Usar:
validar_homogeneidade_colunas(2026)

def validar_outliers_detalhados(ano):
    """Analisa outliers negativos e extremos com contexto."""
    
    con = sqlite3.connect(DB_PATH)
    tabela = tabela_ano(ano)
    
    print(f"Validando outliers detalhados ({ano})\n")
    
    # Valores negativos
    print("="*70)
    print("VALORES NEGATIVOS (Anomalias)")
    print("="*70)
    
    colunas_analise = {
        "INGRESODT": "Ingresso",
        "COSTEDT": "Custo",
        "RENTADT": "Renda"
    }
    
    for coluna_db, nome in colunas_analise.items():
        cursor = con.execute(f"""
            SELECT CODEDT, FENTREGA, TRANSPORTISTA, "{coluna_db}"
            FROM "{tabela}"
            WHERE "{coluna_db}" IS NOT NULL AND CAST("{coluna_db}" AS REAL) < 0
            ORDER BY CAST("{coluna_db}" AS REAL) ASC
            LIMIT 10
        """)
        
        negativos = cursor.fetchall()
        print(f"\n{nome} negativo: {len(negativos)} registos")
        
        for codedt, fentrega, transportista, valor in negativos[:5]:
            print(f"  CODEDT={codedt}, DATA={fentrega}, TRANSP={transportista}, {nome}={float(valor):,.2f}")
    
    # Valores extremamente altos
    print(f"\n{'='*70}")
    print("VALORES EXTREMAMENTE ALTOS")
    print("="*70)
    
    for coluna_db, nome in colunas_analise.items():
        cursor = con.execute(f"""
            SELECT CODEDT, FENTREGA, TRANSPORTISTA, "{coluna_db}"
            FROM "{tabela}"
            WHERE "{coluna_db}" IS NOT NULL 
            AND CAST("{coluna_db}" AS REAL) > 1000
            ORDER BY CAST("{coluna_db}" AS REAL) DESC
            LIMIT 10
        """)
        
        altos = cursor.fetchall()
        print(f"\n{nome} > 1000: {len(altos)} registos")
        
        for codedt, fentrega, transportista, valor in altos[:5]:
            print(f"  CODEDT={codedt}, DATA={fentrega}, TRANSP={transportista}, {nome}={float(valor):,.2f}")
    
    # Resumo por TRANSPORTISTA
    print(f"\n{'='*70}")
    print("RESUMO POR TRANSPORTISTA (Ingressos negativos)")
    print("="*70)
    
    cursor = con.execute(f"""
        SELECT TRANSPORTISTA, COUNT(*) as count, MIN(CAST(INGRESODT AS REAL)) as min_val
        FROM "{tabela}"
        WHERE INGRESODT IS NOT NULL AND CAST(INGRESODT AS REAL) < 0
        GROUP BY TRANSPORTISTA
        ORDER BY count DESC
        LIMIT 10
    """)
    
    for transportista, count, min_val in cursor.fetchall():
        print(f"  {transportista}: {count} registos (mín: {float(min_val):,.2f})")
    
    con.close()


# Usar:
validar_outliers_detalhados(2026)